In [ ]:
# 第1格：依赖（首次执行）
!pip install -q lightgbm pulearn pyarrow pyyaml scikit-learn joblib pandas numpy

In [ ]:
# 第2格：参数（改 TABLE_NAME、MODEL_ROOT）
import sys
from pathlib import Path

import dtools

TABLE_NAME = "ai_decision_dev.fxj_lookalike_pu_training"
MODEL_ROOT = Path(".").resolve()
OUTPUT_PARQUET = MODEL_ROOT / "data" / "training_pu.parquet"

LIMIT_ROWS = None
UNLABELED_SAMPLE_FRAC = 1.0
MS13_MIN_FOR_UNLABELED = 0

sys.path.insert(0, str(MODEL_ROOT / "src"))
print("MODEL_ROOT:", MODEL_ROOT)
print("输出:", OUTPUT_PARQUET)

In [ ]:
# 第3格：导出 parquet
extra = ""
if MS13_MIN_FOR_UNLABELED and MS13_MIN_FOR_UNLABELED > 0:
    extra += f"\n  AND (pu_label = 1 OR ms13_score >= {float(MS13_MIN_FOR_UNLABELED)})"
if UNLABELED_SAMPLE_FRAC < 1.0:
    extra += f"\n  AND (pu_label = 1 OR rand() < {float(UNLABELED_SAMPLE_FRAC)})"
limit_sql = f"\nLIMIT {int(LIMIT_ROWS)}" if LIMIT_ROWS else ""

query = f"""
SELECT *
FROM {TABLE_NAME}
WHERE dataset_split IN ('train', 'val')
{extra}
{limit_sql}
""".strip()

print(query)
df_data = dtools.get_as_frame(query)
print(f"行数: {len(df_data)}, 列数: {len(df_data.columns)}")
print(df_data["pu_label"].value_counts())

OUTPUT_PARQUET.parent.mkdir(parents=True, exist_ok=True)
df_data.to_parquet(OUTPUT_PARQUET, index=False)
print("已保存:", OUTPUT_PARQUET)

In [ ]:
# 第4格：训练（同目录需有 config.yaml 与 src/）
import json

from config_loader import load_config, resolve_path
from dataset import apply_filters, load_table, prepare_splits, subsample_unlabeled, to_xy
from train_pu import (
    save_sklearn_pu_artifacts,
    save_lgbm_artifacts,
    train_elkanoto_pu,
    train_weighted_naive_pu,
)

config_path = MODEL_ROOT / "config.yaml"
cfg = load_config(config_path)
data_path = resolve_path(cfg["data"]["input_path"], config_path)
if not data_path.exists():
    data_path = OUTPUT_PARQUET

df = load_table(data_path)
df = apply_filters(df, cfg)
label_col = cfg["data"]["label_col"]
train_df, val_df, feature_columns = prepare_splits(df, cfg)
train_df = subsample_unlabeled(
    train_df,
    label_col,
    float(cfg["training"].get("unlabeled_subsample_ratio", 1.0)),
    int(cfg["training"]["random_seed"]),
)
x_train, y_train = to_xy(train_df, feature_columns, label_col)
x_val, y_val = to_xy(val_df, feature_columns, label_col)

pu_cfg = cfg["pu"]
train_cfg = cfg["training"]
out_dir = resolve_path(cfg["output"]["artifacts_dir"], config_path)
seed = int(train_cfg["random_seed"])

if pu_cfg.get("method", "elkanoto") == "elkanoto":
    clf, metrics = train_elkanoto_pu(
        x_train, y_train, x_val, y_val,
        params=train_cfg["params"],
        hold_out_ratio=float(pu_cfg.get("hold_out_ratio", 0.1)),
        seed=seed,
    )
    manifest = save_sklearn_pu_artifacts(
        clf, metrics, feature_columns, out_dir, cfg["output"]["model_name"]
    )
else:
    booster, metrics = train_weighted_naive_pu(
        x_train, y_train, x_val, y_val,
        params=dict(train_cfg["params"]),
        unlabeled_weight=float(pu_cfg.get("unlabeled_weight", 0.05)),
        num_boost_round=int(train_cfg["num_boost_round"]),
        early_stopping_rounds=int(train_cfg["early_stopping_rounds"]),
        seed=seed,
    )
    manifest = save_lgbm_artifacts(
        booster, metrics, feature_columns, out_dir, cfg["output"]["model_name"]
    )

print(json.dumps(metrics["val"], ensure_ascii=False, indent=2))
print("模型:", manifest["model_path"])